# FireSight-IR | Module 1c v2 — Surface Context (API-only, no manual downloads)

**Project:** FireSight-IR  
**Author:** Emmanuel Ibekwe | github.com/Ibekwemmanuel7  
**Platform:** Google Colab  
**Depends on:** Module 1b outputs (`viirs_fp_atm_*.parquet`) in Google Drive

---

## What changed from v1

| Feature | v1 (broken) | v2 (this notebook) |
|---|---|---|
| Land cover | ESA CCI via CDS (401 error) | **MODIS MCD12Q1 v6.1 via earthaccess** |
| Persistent emitters | VNP46A1 HDF5 group parse fail | **VNP46A2 monthly composite + OSM fallback** |
| Burn scars | NIFC Geomac (deprecated) | **MTBS (Monitoring Trends in Burn Severity) via earthaccess** |
| auth | CDS key needed | **earthaccess only — same NASA login as Module 1a/1b** |

## MODIS MCD12Q1 vs ESA CCI

For western CONUS wildfire work, MODIS MCD12Q1 is actually better than ESA CCI:
- 500 m resolution (vs 300 m ESA CCI) — close enough for our 375 m VIIRS pixels
- Annual layers 2001–2023 — we can match land cover year to fire year
- IGBP 17-class scheme maps cleanly to our 8-class schema (forest/shrub/grassland/cropland/urban/bare/water/other)
- Downloaded via earthaccess — same auth you already have
- Western CONUS domain covered by MODIS tiles h08v04, h08v05, h09v04, h09v05

## What this module produces

```
data/processed/viirs_fp_srf_v2/viirs_fp_srf_v2_{YEAR}.parquet  ← real land cover
data/processed/patches/firesight_patches.h5                     ← mlp_srf updated
data/raw/surface/modis_lc_*.tif                                 ← cached MODIS tiles
data/raw/surface/mtbs_burn_scars.json                           ← MTBS perimeters
```

---
## Section 0 — Mount and install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
!pip install earthaccess rasterio pyproj requests pandas numpy \
             xarray h5py h5netcdf pyarrow tqdm scipy scikit-learn \
             pyhdf pymodis -q

# gdal is needed to reproject MODIS sinusoidal tiles
!apt-get install -y gdal-bin python3-gdal -q 2>/dev/null

import os, time, warnings, json, requests, subprocess, zipfile, shutil
import numpy as np
import pandas as pd
import h5py
import xarray as xr
import rasterio
from rasterio.merge import merge as rasterio_merge
from rasterio.warp import calculate_default_transform, reproject, Resampling
import scipy.spatial
import earthaccess
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
print('All imports OK.')

---
## Section 1 — Configuration

In [ ]:
BASE_DIR    = '/content/drive/MyDrive/firesight-ir'
ATM_DIR     = f'{BASE_DIR}/data/processed/viirs_fp_atm'
SRF_DIR     = f'{BASE_DIR}/data/processed/viirs_fp_srf_v2'   # new output dir
PATCH_DIR   = f'{BASE_DIR}/data/processed/patches'
SPLIT_DIR   = f'{BASE_DIR}/data/splits'
RAW_SRF_DIR = f'{BASE_DIR}/data/raw/surface'
FIGURES_DIR = f'{BASE_DIR}/figures'

ARCHIVE_PATH = f'{PATCH_DIR}/firesight_patches.h5'

for d in [SRF_DIR, PATCH_DIR, SPLIT_DIR, RAW_SRF_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

# Domain
BBOX_TUPLE = (-125.0, 32.0, -109.0, 49.0)   # west, south, east, north
BBOX_DICT  = {'lon_min':-125.0, 'lon_max':-109.0, 'lat_min':32.0, 'lat_max':49.0}

# Years
TRAIN_YEARS = [2018, 2019, 2020, 2021, 2022]
VAL_YEAR    = 2023
ALL_YEARS   = TRAIN_YEARS + [VAL_YEAR]

PATCH_SIZE = 32

# Labels
LABEL_NONFIRE    = 0
LABEL_WILDFIRE   = 1
LABEL_FALSEALARM = 2

# OSM thresholds
OSM_URBAN_THRESHOLD_KM      = 2.0
OSM_INDUSTRIAL_THRESHOLD_KM = 5.0

# MODIS MCD12Q1 IGBP → our 8-class schema
# IGBP: 1-5=forest types, 6-7=shrub, 8-10=savanna/grassland,
#        11=wetland, 12=cropland, 13=urban, 14=crop/natural mosaic,
#        15=snow/ice, 16=barren, 17=water
MODIS_LC_REMAP = {
    1:  'forest', 2:  'forest', 3:  'forest', 4:  'forest', 5:  'forest',
    6:  'shrub',  7:  'shrub',
    8:  'grassland', 9: 'grassland', 10: 'grassland',
    11: 'water',
    12: 'cropland', 14: 'cropland',
    13: 'urban',
    15: 'bare', 16: 'bare',
    17: 'water',
    0:  'other',  # fill/unclassified
    255: 'other',
}
LC_CLASSES = ['forest','shrub','grassland','cropland','urban','bare','water','other']

# ATM feature columns — must match Module 1b
ATM_FEATURE_COLS = [
    'era5_t2m','era5_pbl','era5_tcwv','era5_sp',
    'era5_t_1000hPa','era5_t_850hPa','era5_t_700hPa','era5_t_500hPa','era5_t_300hPa',
    'era5_q_1000hPa','era5_q_850hPa','era5_q_700hPa','era5_q_500hPa','era5_q_300hPa',
    'beer_lambert_proxy','atm_instability',
]
SRF_FEATURE_COLS = [
    'lc_forest','lc_shrub','lc_grassland','lc_cropland',
    'lc_urban','lc_bare','lc_water','lc_other',
    'dnb_radiance','dnb_is_persistent',
    'dist_urban_km','dist_powerplant_km','dist_industrial_km',
    'is_urban','is_industrial',
    'sol_zen','is_day',
    'is_prior_burn_scar','burn_scar_age_years',
    'firms_type',
]
N_ATM = len(ATM_FEATURE_COLS)  # 16
N_SRF = len(SRF_FEATURE_COLS)  # 20

print('Configuration set.')
print(f'  ATM features: {N_ATM} | SRF features: {N_SRF}')
print(f'  Output dir  : {SRF_DIR}')

---
## Section 2 — Authenticate and load Module 1b data

In [ ]:
auth = earthaccess.login(strategy='interactive')
assert auth.authenticated
print(f'Authenticated: {auth.username}')

In [ ]:
viirs_data = {}
for year in ALL_YEARS:
    fp = f'{ATM_DIR}/viirs_fp_atm_{year}.parquet'
    if os.path.exists(fp):
        df = pd.read_parquet(fp)
        df['date'] = pd.to_datetime(df['date'])
        df['year'] = df['date'].dt.year
        viirs_data[year] = df
        print(f'  {year}: {len(df):>9,} pixels | {len(df.columns)} cols')
    else:
        print(f'  {year}: NOT FOUND — run Module 1b first')

assert viirs_data, 'No Module 1b data found.'

# Verify ATM columns
sample = list(viirs_data.values())[0]
missing = [c for c in ATM_FEATURE_COLS if c not in sample.columns]
if missing:
    print(f'[WARN] Missing ATM cols: {missing}')
    print('Actual era5 cols:', [c for c in sample.columns if 'era5' in c])
else:
    print(f'\n✓ All {N_ATM} ATM columns present')
print(f'Total: {sum(len(v) for v in viirs_data.values()):,} fire pixels')

---
## Section 3 — MODIS MCD12Q1 Land Cover (replaces ESA CCI)

Downloads MODIS land cover tiles for western CONUS via earthaccess.
Western US domain is covered by 4 MODIS tiles: h08v04, h08v05, h09v04, h09v05.

We download the 2020 annual layer (representative for all years in our dataset)
and reproject from MODIS sinusoidal to WGS84 GeoTIFF for point sampling.

**Cached:** only downloads once, reuses on subsequent runs.

In [ ]:
MODIS_LC_PATH   = f'{RAW_SRF_DIR}/modis_lc_2020_western_us.tif'
MODIS_RAW_DIR   = f'{RAW_SRF_DIR}/modis_lc_raw'
os.makedirs(MODIS_RAW_DIR, exist_ok=True)


def download_modis_land_cover(out_path, raw_dir, year=2020):
    """
    Download MODIS MCD12Q1 land cover tiles for western CONUS.
    Tiles: h08v04, h08v05, h09v04, h09v05 (cover lon -125 to -109, lat 32-49)
    Reprojects from MODIS sinusoidal to WGS84 GeoTIFF.
    Returns True if file available.
    """
    if os.path.exists(out_path):
        size_mb = os.path.getsize(out_path) / 1e6
        print(f'  MODIS LC already cached ({size_mb:.1f} MB) — skipping download')
        return True

    print(f'  Searching MCD12Q1 for year {year}...')
    date_str = f'{year}-01-01'

    try:
        results = earthaccess.search_data(
            short_name   = 'MCD12Q1',
            version      = '061',
            bounding_box = (-125.0, 32.0, -109.0, 49.0),
            temporal     = (date_str, f'{year}-12-31'),
            count        = 20,
        )
        if not results:
            print('  No MCD12Q1 granules found — check earthaccess auth')
            return False

        print(f'  Found {len(results)} tiles — downloading...')
        downloaded = earthaccess.download(results, raw_dir)
        print(f'  Downloaded {len(downloaded)} files to {raw_dir}')

        # Find all HDF files
        hdf_files = [f for f in os.listdir(raw_dir) if f.endswith('.hdf')]
        if not hdf_files:
            print('  No HDF files found after download')
            return False

        print(f'  Processing {len(hdf_files)} HDF tiles...')
        reprojected = []

        for hdf_file in hdf_files:
            hdf_path = os.path.join(raw_dir, hdf_file)
            tif_path = hdf_path.replace('.hdf', '_lc.tif')

            # Extract LC_Type1 (IGBP) subdataset and reproject to WGS84
            # MODIS HDF subdataset naming convention
            subdataset = f'HDF4_EOS:EOS_GRID:"{hdf_path}":MCD12Q1:LC_Type1'

            result = subprocess.run([
                'gdalwarp',
                '-t_srs', 'EPSG:4326',
                '-te', '-125.0', '32.0', '-109.0', '49.0',
                '-tr', '0.005', '0.005',   # ~500m in degrees
                '-r', 'near',
                '-overwrite',
                subdataset,
                tif_path
            ], capture_output=True, text=True)

            if result.returncode == 0 and os.path.exists(tif_path):
                reprojected.append(tif_path)
            else:
                print(f'  gdalwarp failed for {hdf_file}: {result.stderr[:200]}')

        if not reprojected:
            print('  No tiles reprojected successfully')
            return False

        # Mosaic all tiles into a single GeoTIFF
        print(f'  Mosaicking {len(reprojected)} tiles...')
        src_files = [rasterio.open(f) for f in reprojected]
        mosaic, transform = rasterio_merge(src_files)

        out_meta = src_files[0].meta.copy()
        out_meta.update({
            'driver': 'GTiff',
            'height': mosaic.shape[1],
            'width':  mosaic.shape[2],
            'transform': transform,
            'crs': 'EPSG:4326',
            'compress': 'lzw',
        })
        with rasterio.open(out_path, 'w', **out_meta) as dst:
            dst.write(mosaic)

        for src in src_files:
            src.close()

        size_mb = os.path.getsize(out_path) / 1e6
        print(f'  ✓ MODIS LC mosaic saved: {out_path} ({size_mb:.1f} MB)')
        return True

    except Exception as e:
        print(f'  MODIS LC download failed: {e}')
        import traceback; traceback.print_exc()
        return False


def assign_modis_land_cover(df, lc_path, lc_remap, lc_classes):
    """
    Sample MODIS MCD12Q1 IGBP land cover at each fire pixel location.
    Adds one-hot lc_* columns and lc_class string column.
    Falls back to zero-fill if file unavailable.
    """
    if not os.path.exists(lc_path):
        print('  [WARN] MODIS LC file not found — lc columns = 0')
        for cls in lc_classes:
            df[f'lc_{cls}'] = np.float32(0)
        df['lc_class'] = 'unknown'
        return df

    lats = df['latitude'].values
    lons = df['longitude'].values

    with rasterio.open(lc_path) as src:
        rows, cols = rasterio.transform.rowcol(src.transform, lons, lats)
        rows = np.clip(rows, 0, src.height - 1)
        cols = np.clip(cols, 0, src.width  - 1)
        r_min, r_max = int(rows.min()), int(rows.max())
        c_min, c_max = int(cols.min()), int(cols.max())
        window = rasterio.windows.Window(
            c_min, r_min, c_max - c_min + 1, r_max - r_min + 1
        )
        data = src.read(1, window=window)
        lc_codes = data[rows - r_min, cols - c_min]

    lc_str = np.array([lc_remap.get(int(c), 'other') for c in lc_codes])
    df['lc_class'] = lc_str
    for cls in lc_classes:
        df[f'lc_{cls}'] = (lc_str == cls).astype(np.float32)

    # Summary
    unique, counts = np.unique(lc_str, return_counts=True)
    print('  LC distribution: ' + ', '.join(f'{c}={n:,}' for c, n in zip(unique, counts)))
    return df


print('MODIS land cover functions defined.')

In [ ]:
print('Downloading MODIS MCD12Q1 land cover...')
lc_available = download_modis_land_cover(MODIS_LC_PATH, MODIS_RAW_DIR, year=2020)
print(f'Land cover available: {lc_available}')

---
## Section 4 — VNP46A2 DNB Monthly Composite (fixed)

VNP46A2 is the gap-filled monthly composite (vs VNP46A1 daily).  
The HDF5 group path is different from VNP46A1 — this version uses the correct path.
Falls back to OSM proximity if streaming fails.

In [ ]:
DNB_CACHE_PATH = f'{RAW_SRF_DIR}/viirs_dnb_2020_western_us.npy'
DNB_META_PATH  = f'{RAW_SRF_DIR}/viirs_dnb_2020_meta.json'
DNB_THRESHOLD  = 5.0   # nW/cm²/sr — Elvidge et al. 2017


def download_vnp46a2_composite(bbox_tuple, cache_path, meta_path, year=2020, month=6):
    """
    Download VNP46A2 gap-filled monthly DNB composite.
    VNP46A2 HDF5 structure differs from VNP46A1 — uses correct group path.
    """
    if os.path.exists(cache_path) and os.path.exists(meta_path):
        print('  DNB composite loaded from cache')
        return np.load(cache_path), json.load(open(meta_path))

    date_str = f'{year}-{month:02d}-01'
    print(f'  Searching VNP46A2 for {date_str}...')

    # VNP46A2 possible group paths (varies by version)
    DNB_GROUP_PATHS = [
        'HDFEOS/GRIDS/VNP_Grid_DNB/Data Fields',
        'HDFEOS/GRIDS/Monthly_moonlight_Corrected_NTL/Data Fields',
        'HDFEOS/GRIDS/VNP_Grid_Monthly_2km_DNB/Data Fields',
    ]
    DNB_VAR_NAMES = [
        'Gap_Filled_DNB_BRDF-Corrected_NTL',
        'DNB_At_Sensor_Radiance_500m',
        'NearNadir_Composite_Snow_Free',
        'AllAngle_Composite_Snow_Free',
    ]

    try:
        results = earthaccess.search_data(
            short_name   = 'VNP46A2',
            bounding_box = bbox_tuple,
            temporal     = (date_str, f'{year}-{month:02d}-28'),
            count        = 20,
        )
        print(f'  Found {len(results)} VNP46A2 granules')
        if not results:
            return None, None

        # FIX: open() returns a list — iterate directly, no context manager
        file_handles = earthaccess.open(results[:6])

        dnb_arrays = []
        for fh in file_handles:
            found = False
            for group in DNB_GROUP_PATHS:
                for var in DNB_VAR_NAMES:
                    try:
                        with warnings.catch_warnings():
                            warnings.simplefilter('ignore')
                            ds = xr.open_dataset(fh, engine='h5netcdf', group=group)
                        if var in ds.data_vars:
                            arr = ds[var].values.astype(np.float32)
                            # Apply scale factor if needed (VNP46A2 uses scale 0.1)
                            if arr.max() > 1e4:
                                arr = arr * 0.1
                            dnb_arrays.append(arr)
                            print(f'    Found {var} in {group}')
                            ds.close()
                            found = True
                            break
                    except Exception:
                        pass
                if found:
                    break

        if not dnb_arrays:
            print('  Could not parse VNP46A2 — DNB will use OSM-only fallback')
            return None, None

        dnb = np.nanmedian(np.stack(dnb_arrays), axis=0).astype(np.float32)
        lon_min, lat_min, lon_max, lat_max = bbox_tuple
        meta = {
            'lat_min': lat_min, 'lat_max': lat_max,
            'lon_min': lon_min, 'lon_max': lon_max,
            'nrows': dnb.shape[0], 'ncols': dnb.shape[1],
        }
        np.save(cache_path, dnb)
        json.dump(meta, open(meta_path, 'w'))
        print(f'  ✓ DNB composite cached: shape={dnb.shape}')
        return dnb, meta

    except Exception as e:
        print(f'  VNP46A2 failed: {e} — DNB features will use OSM fallback')
        return None, None


def assign_dnb(df, dnb_array, meta, threshold=DNB_THRESHOLD):
    """Sample DNB radiance at each fire pixel. Falls back to 0 if unavailable."""
    if dnb_array is None:
        df['dnb_radiance']      = np.float32(0)
        df['dnb_is_persistent'] = np.float32(0)
        return df
    nrows, ncols = meta['nrows'], meta['ncols']
    lats = df['latitude'].values
    lons = df['longitude'].values
    ri = np.clip(((meta['lat_max'] - lats) / (meta['lat_max'] - meta['lat_min']) * nrows).astype(int), 0, nrows-1)
    ci = np.clip(((lons - meta['lon_min']) / (meta['lon_max'] - meta['lon_min']) * ncols).astype(int), 0, ncols-1)
    vals = dnb_array[ri, ci].astype(np.float32)
    vals = np.where(vals < 0, np.nan, vals)
    df['dnb_radiance']      = np.log1p(np.nan_to_num(vals, nan=0.0))
    df['dnb_is_persistent'] = (vals > threshold).astype(np.float32)
    return df


print('DNB functions defined.')

---
## Section 5 — OSM infrastructure (unchanged, known working)

In [ ]:
OSM_CACHE = f'{RAW_SRF_DIR}/osm_features.json'


def download_osm_features(bbox_dict, cache_path):
    if os.path.exists(cache_path):
        results = json.load(open(cache_path))
        for k, v in results.items(): print(f'  OSM {k}: {len(v)} (cache)')
        return results

    overpass_url = 'https://overpass-api.de/api/interpreter'
    s, w, n, e   = (bbox_dict['lat_min'], bbox_dict['lon_min'],
                    bbox_dict['lat_max'], bbox_dict['lon_max'])
    queries = {
        'power_plants': f'[out:json][timeout:60];(node["power"="plant"]({s},{w},{n},{e});way["power"="plant"]({s},{w},{n},{e}););out center;',
        'industrial':   f'[out:json][timeout:90];(way["landuse"="industrial"]({s},{w},{n},{e});node["industrial"~"oil|gas|refinery|chemical"]({s},{w},{n},{e}););out center;',
        'urban':        f'[out:json][timeout:60];(node["place"~"city|town"]({s},{w},{n},{e}););out center;',
    }
    results = {}
    for key, query in queries.items():
        try:
            resp   = requests.get(overpass_url, params={'data': query}, timeout=120)
            coords = []
            if resp.status_code == 200:
                for el in resp.json().get('elements', []):
                    if 'lat' in el:          coords.append((el['lat'],    el['lon']))
                    elif 'center' in el:     coords.append((el['center']['lat'], el['center']['lon']))
            results[key] = coords
            print(f'  OSM {key}: {len(coords)}')
        except Exception as ex:
            print(f'  OSM {key}: error ({ex})')
            results[key] = []
        time.sleep(3)
    json.dump(results, open(cache_path, 'w'))
    return results


def assign_osm_distances(df, osm_features):
    pts = np.column_stack([df['latitude'].values, df['longitude'].values])
    def min_dist_km(pts, ref):
        if not ref: return np.full(len(pts), 999.0, dtype=np.float32)
        tree = scipy.spatial.cKDTree(np.array(ref))
        d, _  = tree.query(pts)
        return (d * 111.0).astype(np.float32)
    df['dist_powerplant_km'] = min_dist_km(pts, osm_features.get('power_plants',[]))
    df['dist_industrial_km'] = min_dist_km(pts, osm_features.get('industrial',  []))
    df['dist_urban_km']      = min_dist_km(pts, osm_features.get('urban',       []))
    df['is_urban']      = (df['dist_urban_km']      <= OSM_URBAN_THRESHOLD_KM      ).astype(np.float32)
    df['is_industrial'] = (df['dist_industrial_km'] <= OSM_INDUSTRIAL_THRESHOLD_KM ).astype(np.float32)
    return df


print('OSM functions defined.')

---
## Section 6 — MTBS Burn Scars (replaces deprecated NIFC Geomac)

MTBS (Monitoring Trends in Burn Severity) provides fire perimeters for all fires
>1000 acres in western US since 1984, available via earthaccess.
These are the highest-quality historical burn perimeters available for CONUS.

In [ ]:
BURN_CACHE = f'{RAW_SRF_DIR}/mtbs_burn_scars.json'


def download_mtbs_burn_perimeters(bbox_dict, cache_path, years_before=2018):
    """
    Download MTBS fire perimeters for western CONUS.
    Uses the USFS ArcGIS REST API (public, no auth required).
    Falls back to NIFC WFIGS if MTBS unavailable.
    """
    if os.path.exists(cache_path):
        records = json.load(open(cache_path))
        print(f'  MTBS burn scars loaded from cache: {len(records)} records')
        return records

    w, s, e, n = (bbox_dict['lon_min'], bbox_dict['lat_min'],
                  bbox_dict['lon_max'], bbox_dict['lat_max'])

    # Try MTBS ArcGIS REST (primary source)
    mtbs_url = ('https://edcintl.cr.usgs.gov/downloads/sciweb1/shared/MTBS_Fire/'
                'data/composite/2022/mtbs_perimeter_data_v2.zip')

    # Fallback: USFS WFIGS public REST (replaced Geomac)
    wfigs_url = ('https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/'
                 'WFIGS_Interagency_Perimeters_YTD/FeatureServer/0/query')

    records = []

    # Strategy 1: WFIGS historical perimeters (most reliable API)
    print('  Trying WFIGS historical perimeters...')
    for start_year in range(1984, years_before, 5):
        end_year = min(start_year + 4, years_before - 1)
        params = {
            'where'    : f"DiscoveryAcres > 1000",
            'geometry' : f'{w},{s},{e},{n}',
            'geometryType': 'esriGeometryEnvelope',
            'spatialRel': 'esriSpatialRelIntersects',
            'outFields': 'DiscoveryAcres,BurnBndLat,BurnBndLon,FireDiscoveryDateTime',
            'returnGeometry': 'false',
            'f': 'json',
            'resultRecordCount': 500,
        }
        try:
            resp = requests.get(wfigs_url, params=params, timeout=30)
            if resp.status_code == 200:
                for feat in resp.json().get('features', []):
                    a = feat.get('attributes', {})
                    lat = a.get('BurnBndLat')
                    lon = a.get('BurnBndLon')
                    if lat and lon:
                        records.append({'lat': float(lat), 'lon': float(lon),
                                        'year': start_year})
        except Exception:
            pass
        time.sleep(0.5)

    # Strategy 2: USGS MTBS query endpoint
    if len(records) < 10:
        print('  Trying USGS MTBS endpoint...')
        mtbs_query = ('https://apps.fs.usda.gov/arcx/rest/services/EDW/'
                      'EDW_MTBS_01/MapServer/0/query')
        params = {
            'where'        : f"Ig_Date < DATE '2018-01-01' AND Acres > 1000",
            'geometry'     : f'{w},{s},{e},{n}',
            'geometryType' : 'esriGeometryEnvelope',
            'spatialRel'   : 'esriSpatialRelIntersects',
            'outFields'    : 'Ig_Date,BurnBndLat,BurnBndLon,Acres',
            'returnGeometry': 'false',
            'f'            : 'json',
            'resultRecordCount': 2000,
        }
        try:
            resp = requests.get(mtbs_query, params=params, timeout=60)
            if resp.status_code == 200:
                feats = resp.json().get('features', [])
                for feat in feats:
                    a   = feat.get('attributes', {})
                    lat = a.get('BurnBndLat')
                    lon = a.get('BurnBndLon')
                    if lat and lon:
                        records.append({'lat': float(lat), 'lon': float(lon),
                                        'year': years_before - 2})
                print(f'  MTBS: {len(records)} total records')
        except Exception as e:
            print(f'  MTBS query failed: {e}')

    json.dump(records, open(cache_path, 'w'))
    print(f'  ✓ Burn scar records saved: {len(records)}')
    return records


def assign_burn_scar_context(df, burn_records, current_year_col='year', radius_km=3.0):
    if not burn_records:
        df['is_prior_burn_scar']  = np.float32(0)
        df['burn_scar_age_years'] = np.float32(0)
        return df
    burn_pts = np.array([[r['lat'], r['lon']] for r in burn_records])
    burn_yrs = np.array([r['year']            for r in burn_records])
    fire_pts = np.column_stack([df['latitude'].values, df['longitude'].values])
    tree     = scipy.spatial.cKDTree(burn_pts)
    dists, idxs = tree.query(fire_pts)
    dist_km  = dists * 111.0
    is_scar  = (dist_km <= radius_km).astype(np.float32)
    scar_age = np.where(is_scar, df[current_year_col].values - burn_yrs[idxs], 0.0).astype(np.float32)
    df['is_prior_burn_scar']  = is_scar
    df['burn_scar_age_years'] = scar_age
    return df


print('MTBS burn scar functions defined.')

---
## Section 7 — Label construction (fixed confidence threshold)

In [ ]:
def construct_labels(df):
    """
    Three-class labelling with real land cover now available.
    False-alarm detection is now enriched by:
      - lc_urban (MODIS) — fires in urban IGBP class
      - dnb_is_persistent — actual DNB emitter if VNP46A2 worked
      - is_industrial (OSM) — within 5km of industrial zone
      - firms_type == 2 — FIRMS explicitly flagged as other static source

    Fix: confidence is integer code 0/1/2 NOT percentage.
    """
    if 'firms_type' not in df.columns:
        df['firms_type'] = np.int8(0)

    df['label'] = np.uint8(LABEL_NONFIRE)

    # Wildfire: confidence >= 1 (nominal or high)
    if 'confidence' in df.columns:
        wildfire_mask = df['confidence'] >= 1
    else:
        wildfire_mask = pd.Series(True, index=df.index)
    df.loc[wildfire_mask, 'label'] = LABEL_WILDFIRE

    # False-alarm: any of these conditions
    fa_firms     = (df['firms_type'] == 2)
    fa_dnb_urban = (df['dnb_is_persistent'] == 1) & \
                   ((df['is_industrial'] == 1) | (df['is_urban'] == 1))
    # With real MODIS land cover: urban IGBP class fires are suspicious
    fa_modis_urban = (df.get('lc_urban', pd.Series(0, index=df.index)) == 1) & \
                     (df['is_industrial'] == 1)

    df.loc[fa_firms | fa_dnb_urban | fa_modis_urban, 'label'] = LABEL_FALSEALARM

    counts = df['label'].value_counts().sort_index()
    total  = len(df)
    for lbl, name in [(0,'non-fire'), (1,'wildfire'), (2,'false-alarm')]:
        n = counts.get(lbl, 0)
        print(f'    {lbl} {name:12s}: {n:>9,}  ({100*n/total:.1f}%)')
    return df


print('Label construction defined.')

---
## Section 8 — Download all surface sources

In [ ]:
print('1. MODIS MCD12Q1 land cover')
lc_ok = download_modis_land_cover(MODIS_LC_PATH, MODIS_RAW_DIR, year=2020)

print('\n2. VNP46A2 DNB nighttime lights')
dnb_array, dnb_meta = download_vnp46a2_composite(
    BBOX_TUPLE, DNB_CACHE_PATH, DNB_META_PATH
)
print(f'   DNB available: {dnb_array is not None}')

print('\n3. OSM infrastructure')
osm_features = download_osm_features(BBOX_DICT, OSM_CACHE)

print('\n4. MTBS historical burn scars')
burn_records = download_mtbs_burn_perimeters(BBOX_DICT, BURN_CACHE, years_before=2018)

print('\n✓ All surface sources ready')
print(f'  Land cover   : {"MODIS MCD12Q1" if lc_ok else "zero-filled (download failed)"}')
print(f'  DNB          : {"VNP46A2" if dnb_array is not None else "OSM fallback"}')
print(f'  OSM features : {sum(len(v) for v in osm_features.values())} total')
print(f'  Burn scars   : {len(burn_records)} perimeters')

---
## Section 9 — Apply surface enrichment to all years

In [ ]:
enriched_srf = {}

for year in ALL_YEARS:
    out_path = f'{SRF_DIR}/viirs_fp_srf_v2_{year}.parquet'

    if os.path.exists(out_path):
        df = pd.read_parquet(out_path)
        enriched_srf[year] = df
        print(f'  {year}: loaded ({len(df):,} pixels, {len(df.columns)} cols)')
        continue

    if year not in viirs_data:
        print(f'  {year}: missing — skip')
        continue

    print(f'\n── {year} ──')
    df = viirs_data[year].copy()
    if 'firms_type' not in df.columns:
        df['firms_type'] = np.int8(0)

    print('  MODIS land cover...')
    df = assign_modis_land_cover(df, MODIS_LC_PATH, MODIS_LC_REMAP, LC_CLASSES)

    print('  DNB...')
    df = assign_dnb(df, dnb_array, dnb_meta)

    print('  OSM distances...')
    df = assign_osm_distances(df, osm_features)

    print('  Burn scars...')
    df = assign_burn_scar_context(df, burn_records, current_year_col='year')

    for col in ['sol_zen','is_day']:
        if col not in df.columns:
            df[col] = np.float32(0)

    print('  Labels...')
    df = construct_labels(df)

    df.to_parquet(out_path, index=False)
    enriched_srf[year] = df
    print(f'  ✓ {year}: {len(df):,} pixels, {len(df.columns)} cols → Drive')

print('\n✓ Surface enrichment complete')

---
## Section 10 — Feature audit

In [ ]:
sample = list(enriched_srf.values())[0]
all_feat_cols = ATM_FEATURE_COLS + SRF_FEATURE_COLS

print(f'Feature audit ({len(all_feat_cols)} features):')
print(f'{"Column":<35} {"Mean|val|":>12} {"NonZero%":>10} {"Status"}')
print('-' * 65)
real_count = 0
zero_count = 0
for col in all_feat_cols:
    if col not in sample.columns:
        print(f'  ✗ {col:<33} MISSING')
        continue
    vals     = sample[col].values.astype(np.float64)
    mean_abs = np.nanmean(np.abs(vals))
    nonzero  = 100 * (vals != 0).mean()
    if mean_abs < 1e-6 and nonzero < 0.1:
        status = 'ZERO'
        zero_count += 1
        icon = '○'
    else:
        status = 'REAL'
        real_count += 1
        icon = '✓'
    print(f'  {icon} {col:<33} {mean_abs:>12.4f} {nonzero:>9.1f}%  {status}')

print(f'\n  {real_count} real features | {zero_count} zero-filled')
if real_count >= 30:
    print('  ✓ Sufficient real features for training')
else:
    print('  [WARN] Many features still zero-filled — check downloads above')

---
## Section 11 — Update HDF5 archive with new surface features

Now that land cover is real, we update the `mlp_srf` dataset in the existing
HDF5 archive in-place. This replaces the zero-filled land cover columns.

In [ ]:
def update_hdf5_srf_features(archive_path, enriched_srf, srf_feature_cols):
    """
    Update mlp_srf dataset in-place with real surface features.
    Matches records by (year, latitude, longitude) triple.
    """
    with h5py.File(archive_path, 'a') as hf:
        N = hf['labels'].shape[0]
        print(f'Archive: {N:,} patches')

        years_arr = hf['meta/year'][:]
        lats_arr  = hf['meta/center_lat'][:]
        lons_arr  = hf['meta/center_lon'][:]

        for year, df in tqdm(enriched_srf.items(), desc='Updating SRF features'):
            # Get archive indices for this year
            year_mask   = (years_arr == year)
            arch_indices = np.where(year_mask)[0]

            if len(arch_indices) == 0:
                print(f'  {year}: no archive entries found — skipping')
                continue

            # Fill missing SRF columns with 0
            for col in srf_feature_cols:
                if col not in df.columns:
                    df[col] = np.float32(0)

            srf_vals = df[srf_feature_cols].values.astype(np.float32)
            srf_vals = np.nan_to_num(srf_vals, nan=0.0)

            # Match archive order using KDTree on lat/lon
            arch_lats = lats_arr[arch_indices]
            arch_lons = lons_arr[arch_indices]
            df_pts   = np.column_stack([df['latitude'].values, df['longitude'].values])
            arch_pts = np.column_stack([arch_lats, arch_lons])

            tree      = scipy.spatial.cKDTree(df_pts)
            _, match  = tree.query(arch_pts)

            matched_srf = srf_vals[match]  # (n_year_patches, N_SRF)

            # Write in sorted order for HDF5 efficiency
            sort_order = np.argsort(arch_indices)
            sorted_indices = arch_indices[sort_order]
            sorted_srf     = matched_srf[sort_order]

            # Update in chunks
            chunk = 10000
            for i in range(0, len(sorted_indices), chunk):
                sl_idx = sorted_indices[i:i+chunk]
                sl_srf = sorted_srf[i:i+chunk]
                hf['mlp_srf'][sl_idx] = sl_srf

            print(f'  {year}: updated {len(arch_indices):,} SRF vectors')

    print('\n✓ HDF5 mlp_srf dataset updated')


print('HDF5 update function defined.')

In [ ]:
if os.path.exists(ARCHIVE_PATH):
    print(f'Updating HDF5 archive SRF features...')
    update_hdf5_srf_features(ARCHIVE_PATH, enriched_srf, SRF_FEATURE_COLS)
else:
    print(f'[WARN] Archive not found: {ARCHIVE_PATH}')
    print('  Run Module 1c first to create the archive, then rerun this cell.')

---
## Section 12 — Verify and summary

In [ ]:
# Quick verification
print('=== Surface Feature Verification ===')
all_years_df = pd.concat(list(enriched_srf.values()), ignore_index=True)

lc_cols = [f'lc_{c}' for c in LC_CLASSES]
lc_coverage = {col: (all_years_df[col] > 0).mean() for col in lc_cols if col in all_years_df.columns}
print('\nLand cover class coverage (% of pixels assigned):')
for col, pct in sorted(lc_coverage.items(), key=lambda x: -x[1]):
    bar = '█' * int(pct * 40)
    print(f'  {col:<18} {pct*100:5.1f}%  {bar}')

print('\nLabel distribution (all years):')
counts = all_years_df['label'].value_counts().sort_index()
total  = len(all_years_df)
for lbl, name in [(0,'non-fire'),(1,'wildfire'),(2,'false-alarm')]:
    n = counts.get(lbl, 0)
    print(f'  {lbl} {name:12s}: {n:>9,} ({100*n/total:.1f}%)')

print('\n=== Module 1c v2 Complete ===')
print(f'  Output : {SRF_DIR}')
print(f'  Archive: {ARCHIVE_PATH} (mlp_srf updated in-place)')
print(f'\n  Next: Re-run Module 2 to update scalers and class weights,')
print(f'        then delete the cache in Module 3a and retrain.')